In [1]:
# Cell 1 — Install & imports
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
    "vllm", "transformers", "accelerate", "sentence-transformers",
    "textstat", "scipy", "tqdm"], check=True)

import json, csv, time, re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from vllm import LLM
from transformers import AutoTokenizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.2/279.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.0 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.

In [2]:
# Cell 2 — Paths & config
ARMORM_PATH = "/kaggle/input/datasets/letemoin/armormllama3x8b"
POOL_PATH = "/kaggle/input/datasets/pradeep01223/alpacaeval-500x256-llama8b/pool_candidates_merged_300.jsonl"

OUT_CSV = Path("/kaggle/working/armorm_cache_500.csv")
SELECTIONS_CSV = Path("/kaggle/working/selections_armorm_500.csv")
SUMMARY_CSV = Path("/kaggle/working/armorm_summary_500.csv")

N_LIST = [1, 4, 16, 64, 256]
SCORE_BATCH = 32
RESUME = True

In [6]:
# Cell 3 — Load ArmoRM via vLLM pooling API
tokenizer = AutoTokenizer.from_pretrained(ARMORM_PATH, trust_remote_code=True)

llm = LLM(
    model=ARMORM_PATH,
    runner="pooling",              # pooling / classification head, not generation
    trust_remote_code=True,
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    tensor_parallel_size=1,
)

def build_chat_text(prompt, response):
    chat = [{"role": "user", "content": prompt},
            {"role": "assistant", "content": response}]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)

def armorm_score_batch(pairs):
    texts = [build_chat_text(p, r) for p, r in pairs]
    outputs = llm.encode(texts)
    # ArmoRM returns a multi-objective vector; take the aggregated scalar (last dim / gating output)
    scores = [float(o.outputs.data[-1]) if hasattr(o.outputs, "data") else float(o.outputs.embedding[-1]) for o in outputs]
    return scores

print("ArmoRM loaded via vLLM.")

INFO 07-03 07:35:18 [api_utils.py:273] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': '/kaggle/input/datasets/letemoin/armormllama3x8b'}
WARNING 07-03 07:35:18 [dynamic_module.py:62] Unable to load modeling_custom.LlamaForRewardModelWithGating from /kaggle/input/datasets/letemoin/armormllama3x8b on HF Hub.
WARNING 07-03 07:35:18 [dynamic_module.py:62] Traceback (most recent call last):
WARNING 07-03 07:35:18 [dynamic_module.py:62]   File "/usr/local/lib/python3.12/dist-packages/vllm/transformers_utils/dynamic_module.py", line 44, in try_get_class_from_dynamic_module
WARNING 07-03 07:35:18 [dynamic_module.py:62]     return get_class_from_dynamic_module(
WARNING 07-03 07:35:18 [dynamic_module.py:62]            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
WARNING 07-03 07:35:18 [dynamic_module.py:62]   File "/usr/local/lib/python3.12/dist-packages/transformers/dynamic_modul

ValidationError: 1 validation error for ModelConfig
  Value error, Model architectures ['LlamaForRewardModelWithGating'] are not supported for now. Supported architectures: dict_keys(['AfmoeForCausalLM', 'ApertusForCausalLM', 'AquilaModel', 'AquilaForCausalLM', 'ArceeForCausalLM', 'ArcticForCausalLM', 'AXK1ForCausalLM', 'BaiChuanForCausalLM', 'BaichuanForCausalLM', 'BailingMoeForCausalLM', 'BailingMoeV2ForCausalLM', 'BailingMoeV2_5ForCausalLM', 'BloomForCausalLM', 'ChatGLMModel', 'ChatGLMForConditionalGeneration', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CwmForCausalLM', 'DbrxForCausalLM', 'DeciLMForCausalLM', 'DeepseekForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'ExaoneForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoEForCausalLM', 'Fairseq2LlamaForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FalconH1ForCausalLM', 'FlexOlmoForCau

In [ ]:
# Cell 4 — Helper metrics
KEYWORD_SET = {"effective","important","key","great","best","helpful","excellent",
"significant","valuable","essential","improve","benefit","result","optimize",
"performance","crucial","critical","powerful","robust","strong","useful",
"efficient","successful","positive","major","notable","remarkable","outstanding",
"superior","ideal","optimal","productive","impactful","meaningful","achieve",
"enhance","boost","increase","maximize","support","enable","ensure","provide","offer"}

def keyword_density(text):
    words = re.findall(r"\w+", text.lower())
    return sum(1 for w in words if w in KEYWORD_SET) / len(words) if words else 0.0

def response_length(text):
    return len(text.split())

def repetition_score(text):
    tokens = text.lower().split()
    if len(tokens) < 2: return 0.0
    bigrams = list(zip(tokens, tokens[1:]))
    return 1 - len(set(bigrams)) / len(bigrams)

import textstat
def flesch_score(text):
    try: return textstat.flesch_reading_ease(text)
    except: return 50.0

In [ ]:
# Cell 5 — Score the full pool 
records = []
with open(POOL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"Loaded {len(records)} prompts, {len(records[0]['candidates'])} candidates each.")

done = set()
if RESUME and OUT_CSV.exists():
    with open(OUT_CSV) as f:
        for row in csv.DictReader(f):
            done.add((row["promptid"], row["candidx"]))
    print(f"Resuming — {len(done)} rows already scored.")

write_header = not OUT_CSV.exists() or len(done) == 0
csv_file = open(OUT_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(csv_file, fieldnames=["promptid","candidx","armorm_raw","text"])
if write_header: writer.writeheader()

session_start = time.time()
total_scored = 0

for rec in tqdm(records, desc="ArmoRM pre-scoring"):
    pid = rec["promptid"]
    prompt = rec["instruction"]
    cands = rec["candidates"]

    todo_idx = [i for i in range(len(cands)) if (str(pid), str(i)) not in done]
    if not todo_idx:
        continue

    for b in range(0, len(todo_idx), SCORE_BATCH):
        idx_batch = todo_idx[b:b+SCORE_BATCH]
        pairs = [(prompt, cands[i]) for i in idx_batch]
        scores = armorm_score_batch(pairs)
        for i, s in zip(idx_batch, scores):
            writer.writerow({"promptid": pid, "candidx": i,
                              "armorm_raw": round(s, 6), "text": cands[i]})
        csv_file.flush()
        total_scored += len(idx_batch)

csv_file.close()
print(f"Done. Saved to {OUT_CSV}, total new rows scored: {total_scored}")

In [ ]:
# Cell 6 — N-sweep Best-of-N selection
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

sts_model = SentenceTransformer("all-MiniLM-L6-v2")
def sts_score(text, reference):
    e1 = sts_model.encode(text); e2 = sts_model.encode(reference)
    return float(1.0 - cosine(e1, e2))

cache_df = pd.read_csv(OUT_CSV)
score_cache = {}
for pid, grp in cache_df.groupby("promptid"):
    grp_sorted = grp.sort_values("candidx")
    score_cache[str(pid)] = grp_sorted["armorm_raw"].tolist()

rows = []
for rec in tqdm(records, desc="N sweep"):
    pid = str(rec["promptid"])
    prompt = rec["instruction"]
    cands = rec["candidates"]
    reference = cands[0]
    scores = score_cache[pid]

    for N in N_LIST:
        pool_n = cands[:N]
        scores_n = scores[:N]
        best_idx = int(np.argmax(scores_n))
        best_text = pool_n[best_idx]
        rows.append({
            "promptid": pid, "N": N,
            "armorm_score": scores_n[best_idx],
            "sts": sts_score(best_text, reference),
            "flesch": flesch_score(best_text),
            "resplen": response_length(best_text),
            "kd": keyword_density(best_text),
            "repscore": repetition_score(best_text),
            "selected_text": best_text,
        })

df = pd.DataFrame(rows)
df.to_csv(SELECTIONS_CSV, index=False)
print(f"Saved {SELECTIONS_CSV}: {len(df)} rows")

In [ ]:
# Cell 7 — Aggregate + Type I/II/III checks 
agg = df.groupby("N").agg(
    armorm_mean=("armorm_score","mean"),
    sts_mean=("sts","mean"),
    flesch_mean=("flesch","mean"),
    resplen_mean=("resplen","mean"),
    kd_mean=("kd","mean"),
).round(4)

base_r = agg.loc[1, "armorm_mean"]
eps = 1e-9
agg["ERR"] = ((agg["armorm_mean"] - base_r) / (abs(base_r) + eps)).round(4)
print(agg)

arm_delta = agg.loc[256, "armorm_mean"] - base_r
sts_delta = agg.loc[256, "sts_mean"] - agg.loc[1, "sts_mean"]
print(f"ArmoRM gain N1→N256: {arm_delta:.4f}")
print(f"STS delta N1→N256: {sts_delta:.4f}")

if arm_delta > 0 and sts_delta < -0.003:
    print("TYPE III CONFIRMED — ArmoRM gain with STS decline at scale")
elif arm_delta > 0 and sts_delta < 0:
    print("TYPE III PARTIAL — directional STS decline, check significance")
else:
    print("TYPE III NOT ACTIVATED")

agg.to_csv(SUMMARY_CSV)

In [ ]:
# Cell 8 — Paired significance test
from scipy.stats import ttest_rel

sts_n1 = df[df.N==1].set_index("promptid")["sts"]
sts_n256 = df[df.N==256].set_index("promptid")["sts"]
common = sts_n1.index.intersection(sts_n256.index)

t, p = ttest_rel(sts_n256.loc[common], sts_n1.loc[common])
pooled_std = np.sqrt((sts_n1.loc[common].var() + sts_n256.loc[common].var())/2)
d = (sts_n1.loc[common].mean() - sts_n256.loc[common].mean()) / (pooled_std + eps)

print(f"Paired t-test STS N256 vs N1, n={len(common)}")
print(f"t={t:.3f}, p={p:.4f}, Cohen's d={d:.3f}")

In [ ]:
# Cell 9 — Cond A/B/C dimension-penalty re-selection at N=256
def reward_cond_B(armorm_raw, text):
    # Penalize excessive repetition
    rep = repetition_score(text)
    return armorm_raw - 5.0 * max(0.0, rep - 0.12)

def reward_cond_C(armorm_raw, text):
    # Penalize excessive verbosity (length)
    n = response_length(text)
    return armorm_raw - 0.05 * max(0.0, n - 120)

cond_rows = []
for rec in tqdm(records, desc="Cond A/B/C re-selection at N=256"):
    pid = str(rec["promptid"])
    reference = rec["candidates"][0]
    cands = rec["candidates"][:256]
    scores = score_cache[pid][:256]

    for cond_name, rfn in [
        ("CondA_PureArmoRM", lambda r, t: r),
        ("CondB_RepPenalty", reward_cond_B),
        ("CondC_VerbPenalty", reward_cond_C),
    ]:
        adj_scores = [rfn(s, c) for s, c in zip(scores, cands)]
        best_idx = int(np.argmax(adj_scores))
        best_text = cands[best_idx]

        cond_rows.append({
            "promptid": pid,
            "condition": cond_name,
            "armorm_raw": scores[best_idx],
            "adj_reward": adj_scores[best_idx],
            "sts": sts_score(best_text, reference),
            "flesch": flesch_score(best_text),
            "resplen": response_length(best_text),
            "repscore": repetition_score(best_text),
            "kd": keyword_density(best_text),
        })

cond_df = pd.DataFrame(cond_rows)

cond_summary = cond_df.groupby("condition").agg(
    armorm_raw_mean=("armorm_raw","mean"),
    sts_mean=("sts","mean"),
    flesch_mean=("flesch","mean"),
    resplen_mean=("resplen","mean"),
    repscore_mean=("repscore","mean"),
    kd_mean=("kd","mean"),
).round(4)

print("--- Cond A/B/C Summary at N=256 ---")
print(cond_summary.to_string())

cond_df.to_csv("/kaggle/working/armorm_conds_N256_500.csv", index=False)
cond_summary.to_csv("/kaggle/working/armorm_conds_summary_500.csv")
print("Saved — armorm_conds_N256_500.csv, armorm_conds_summary_500.csv")

In [ ]:
# Cell 10 — Escape route detection (Type II verdict)
condA = cond_summary.loc["CondA_PureArmoRM"]
condB = cond_summary.loc["CondB_RepPenalty"]
condC = cond_summary.loc["CondC_VerbPenalty"]

print("="*60)
print("TYPE II — Penalization Escape Migration (ArmoRM, N=500 pool)")
print("="*60)

routes = {
    "RepPenalty→Length↑": condB["resplen_mean"] > condA["resplen_mean"],
    "RepPenalty→KD↑": condB["kd_mean"] > condA["kd_mean"],
    "VerbPenalty→Rep↑": condC["repscore_mean"] > condA["repscore_mean"],
    "VerbPenalty→KD↑": condC["kd_mean"] > condA["kd_mean"],
    "VerbPenalty→Flesch shift": abs(condC["flesch_mean"] - condA["flesch_mean"]) > 1.0,
}

for route, detected in routes.items():
    print(f"  {route:30s} {'DETECTED' if detected else '-'}")

active_routes = [r for r, v in routes.items() if v]
type2_confirmed = len(active_routes) >= 1

if type2_confirmed:
    print(f"\nTYPE II CONFIRMED — Escape routes: {active_routes}")
else:
    print("\nTYPE II NOT ACTIVATED — no bidirectional escape detected under explicit penalties")